In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained('Salesforce/codegen-350M-mono')

model = AutoModelForCausalLM.from_pretrained('Salesforce/codegen-350M-mono').to(device)

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Este tipo de prompt está pensado para modelos ajustados por instrucciones (instruction-tuned)
text = 'Import re and define a regular expression that matches an email address.'

# Los GPT preentrenados suelen funcionar mejor con generación basada en contexto
# text = 'for i in range(10):'
input_ids = tokenizer(text, return_tensors='pt').input_ids.to(device)

generated_ids = model.generate(
    input_ids,
    max_length=128,
    temperature=1.4,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Import re and define a regular expression that matches an email address.

For Example:  Add an in_message() decorator that accepts a string as parameter, like '(example@example.test).', but return (match_result is replaced by 1). The value of match_result may vary per your data collection or filter. See below for an example of the data filtering process.


# ------------ EXAMPLES --------------

>In test emails (10 emails), you should have "example.test.johndoe@google.com" as match result. You should make use of '


Si interactuamos con un LLM esperando que escriba código como lo harían ChatGPT, Claude o Gemini, podríamos redactar un *prompt* similar al de la variable `text` (por ejemplo: *"importar re y definir una expresión regular que coincida con una dirección de correo electrónico"*). Podemos usar esto como estímulo o entrada para el método `model.generate()` y obtener un resultado que, aunque quizá no sea del todo exacto, probablemente guarde relevancia con el contexto.

Sin embargo, para lograr que un modelo responda de forma natural y directa a este tipo de instrucciones, se requiere un proceso de entrenamiento especializado y adicional conocido como *Instruction Tuning*.

In [4]:
import os
import nbformat
import subprocess

In [5]:
# Descargar el repositorio de GitHub
repo_url = 'https://github.com/mikexcohen/Calculus_book.git'
repo_dir = 'Calculus_book'
subprocess.run(['git', 'clone', repo_url, repo_dir])

# Inicializar una lista para almacenar todos los tokens
all_tokens = []

# Buscar todos los archivos .ipynb en el repositorio
for root, dirs, files in os.walk(repo_dir):
    for file in files:
        if file.endswith('.ipynb'):

            # Cargar y procesar cada cuaderno
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:

                # Analizar la estructura del notebook
                nb_data = nbformat.read(f, as_version=4)

                # Recolectar todas las celdas de código en una lista
                code_cells = []
                for cell in nb_data['cells']:
                    if cell['cell_type'] == 'code':  # La celda contiene código
                        code_cells.append(cell['source'])  # Código fuente de la celda

                # Unir todas las celdas en un solo texto continuo
                all_code = '\n'.join(code_cells)

                # Tokenizar el código y agregarlo a la lista global de tokens
                all_tokens += tokenizer(all_code)['input_ids']

# Verificar el tamaño final del conjunto de datos de entrenamiento
print(f'\n\nEl conjunto de datos de entrenamiento contiene {len(all_tokens):,} tokens, de los cuales {len(set(all_tokens)):,} son únicos.')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8261 > 2048). Running this sequence through the model will result in indexing errors




El conjunto de datos de entrenamiento contiene 166,857 tokens, de los cuales 3,137 son únicos.


In [6]:
# Necesitamos que los tokens tengan formato de tensor
all_tokens = torch.tensor(all_tokens, dtype=torch.long)

In [7]:
# Hyperparametros de entrenamiento
seq_len     = 128 # max longitud de secuencia
batch_size  = 64
num_samples = 200

Clipping Gradient : recorte de gradientes (clipping) es una herramienta fundamental en el entrenamiento de modelos de lenguaje: actúa como un freno de emergencia cuando un lote de datos provoca un pico imprevisto de los gradientes, evitando que los valores se disparen hacia el infinito y se conviertan en

In [8]:
# 1. Forzar precisión completa de 32 bits para máxima estabilidad numérica
model.float()

# 2. Usar una tasa de aprendizaje ultraconservadora
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)

losses = np.zeros(num_samples)

model.train()
for sampli in range(num_samples):

    # Obtener un lote de datos
    ix = torch.randint(len(all_tokens)-seq_len, size=(batch_size,))
    X = all_tokens[ix[:,None] + torch.arange(seq_len)].to(device)

    model.zero_grad()
    outputs = model(X, labels=X)
    loss = outputs.loss

    # CONTROL DE SEGURIDAD: Si el lote da NaN, se omite para no contaminar el modelo
    if torch.isnan(loss):
        continue

    # Retropropagación
    loss.backward()

    # Recorte de gradientes estricto
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

    optimizer.step()
    losses[sampli] = loss.item()

    # Actualizar la visualización del progreso
    if sampli % 42 == 0:
        print(f'Muestra {sampli:4}/{num_samples}, pérdida: {losses[sampli]:.4f}')

Muestra    0/200, pérdida: 1.7587
Muestra   42/200, pérdida: 1.3077
Muestra   84/200, pérdida: 1.1867
Muestra  126/200, pérdida: 1.2295
Muestra  168/200, pérdida: 1.1879


In [10]:
# Evaluacion puramente cualitativa

### Evaluar

In [9]:
text = 'for i in range(10):'
input_ids = tokenizer(text, return_tensors='pt').input_ids.to(device)

generated_ids = model.generate(input_ids, max_length=222, temperature=1.4, do_sample=True)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for i in range(10):
h = 3*np.log(i+1)/10

# draw some grid points
xi = [-.5,.2,.2,.5]
yi = [f3r1(x[0],caulex)[0] for xi in xi ]+.1
ys = [ylim1f(x[i//2,...,1],cauly[i//2,...,1,None],caulex[i//2,...,1])[2].squeeze() for i in xi].\
    sort() *h


# initialise figure and subplots
f,axs = plt.subplots(1,2,figsize=(10,6))
axs[0].text(0.99,.04,greek_letters["Alpha"] ,horizontalalignment='center',color=[.5,.5,.5],verticalalignment='center',weight='bold',size=20)
axs


Aunque la sintaxis de código como Python pueda parecer distinta a los textos en inglés, para un modelo lingüístico no hay diferencia: se trata simplemente de una secuencia de tokens aprendida. Sin embargo, el código presenta particularidades interesantes: utiliza un conjunto de tokens relativamente reducido en comparación con el lenguaje humano y sus fichas suelen tener menos caracteres, lo que representa un reto adicional para que el modelo aprenda a codificar con precisión.

Por esta razón, empresas como OpenAI, Salesforce, Google y Anthropic han invertido un enorme esfuerzo en optimizar sus sistemas para entender y producir software. Ser un buen programador no es algo que surja de forma natural en los modelos de lenguaje generales, sino el resultado de un entrenamiento específico. Lo positivo es que, cuando estas tecnologías se ponen a nuestra disposición, podemos aprovecharlas descargándolas, aplicando *fine-tuning* y adaptándolas a nuestras propias necesidades.